# DAMPED Cosinor analysis on May data!

In [1]:
from circadian_cell_painting import(
    detect,
    OscillationResults,
    cosinor_fitted_long
)
from circadian_cell_painting.osc_detection import DampedCosinorDetection

from pathlib import Path
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
from natsort import natsorted
import matplotlib.colors as mcolors

dataset = Path("/home/derfelt/PartchLabFiles/Circadian "
                    "Rhythms/Live Study/enVision/enVision_pre"
                    "processed/2026-05-11._.PREPROCESSED.parquet")

lf = pl.scan_parquet(dataset).filter(pl.col("feature") == "CalcResultI")

tpoints = lf.select("Timepoint").unique().sort(by="Timepoint").collect().to_series().to_list()

platemap = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026-05-11"
                  "Echo_transfer.xlsx", engine='openpyxl')
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "sample_type"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
).rename({"384_Well": "Well"})

print(tpoints)
display(lf.head(2).collect())
display(platemap)

[24.0, 24.5, 25.0, 25.5, 26.0, 26.5, 27.0, 27.5, 28.0, 28.5, 29.0, 29.5, 30.0, 30.5, 31.0, 31.5, 32.0, 32.5, 33.0, 33.5, 34.0, 34.5, 35.0, 35.5, 36.0, 36.5, 37.0, 37.5, 38.0, 38.5, 39.0, 39.5, 40.0, 40.5, 41.0, 41.5, 42.0, 42.5, 43.0, 43.5, 44.0, 44.5, 45.0, 45.5, 46.0, 46.5, 47.0, 47.5, 48.0, 48.5, 49.0, 49.5, 50.0, 50.5, 51.0, 51.5, 52.0, 52.5, 53.0, 53.5, 54.0, 54.5, 55.0, 55.5, 56.0, 56.5, 57.0, 57.5, 58.0, 58.5, 59.0, 59.5, 60.0, 60.5, 61.0, 61.5, 62.0, 62.5, 63.0, 63.5, 64.0, 64.5, 65.0, 65.5, 66.0, 66.5, 67.0, 67.5, 68.0, 68.5, 69.0, 69.5, 70.0, 70.5, 71.0, 71.5, 72.0, 72.5, 73.0, 73.5, 74.0, 74.5, 75.0, 75.5, 76.0, 76.5, 77.0, 77.5, 78.0, 78.5, 79.0, 79.5, 80.0, 80.5, 81.0, 81.5, 82.0, 82.5, 83.0, 83.5, 84.0, 84.5, 85.0, 85.5, 86.0, 86.5, 87.0, 87.5, 88.0, 88.5, 89.0, 89.5, 90.0, 90.5, 91.0, 91.5, 92.0, 92.5, 93.0, 93.5, 94.0, 94.5, 95.0, 95.5, 96.0, 96.5, 97.0, 97.5, 98.0, 98.5, 99.0, 99.5, 100.0, 100.5, 101.0, 101.5, 102.0, 102.5, 103.0, 103.5, 104.0, 104.5, 105.0, 105.5, 106

Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",24.0,"""CalcResultI""",-5.48105,-5.48105,null,-5.48105,-5.48105,-5.48105,-5.48105,-5.48105,-5.48105,-5.48105
"""A01""",24.5,"""CalcResultI""",-5.4394,-5.4394,null,-5.4394,-5.4394,-5.4394,-5.4394,-5.4394,-5.4394,-5.4394


Well,MoleculeID,strain,sample_type
str,str,str,str
"""G11""","""0.0625% DMSO""","""bmal1""","""dex"""
"""I14""","""Dex_PhenoVue_per2""","""per2""",null
"""J18""","""MK-8800 825nM""","""per2""","""dex"""
"""G07""","""0.25% DMSO""","""bmal1""","""dex"""
"""F19""","""MK-8692 825nM""","""bmal1""","""dex"""
…,…,…,…
"""L20""","""0.0078% DMSO""","""per2""","""dex"""
"""L07""","""0.25% DMSO""","""per2""","""dex"""
"""G17""","""0.0156% DMSO""","""bmal1""","""dex"""


In [6]:
out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/cosinor_analysis")
out_pickle = Path("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/enVision/enVision_pickle")

out_file = out_file / "2026-05-11._.RESULTS.parquet"
out_pickle = out_pickle / "2026-05-11._.obj.pickle"

model = DampedCosinorDetection(
    period=24,
    intervals=1
)

result: OscillationBase = detect(
    file_path=dataset,
    platemap_path=platemap,
    feature_col="feature",
    well_column="Well",
    timepoint_col="Timepoint",
    value_col="mean",
    stat="mean",
    out_file=out_file,
    correction_group=["MoleculeID", "strain"],
    detector=model
)

result.to_pickle(out_pickle)

Well progress:   0%|          | 0/384 [00:00<?, ?it/s]

Well progress: 100%|██████████| 384/384 [00:38<00:00,  9.85it/s]


In [3]:
out_pickle = Path("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/enVision/enVision_pickle")

out_pickle = out_pickle / "2026-05-11._.obj.pickle"

result = OscillationResults.load_pickle(out_pickle)

detrended_data = result.scan_data()
cosinor_data = result.scan_results().filter(pl.col("feature") == "CalcResultI")

well_columns = "Well"
wells = cosinor_data.select(well_columns).unique().sort(well_columns).collect()[well_columns].to_list()

tpoint_col = "Timepoint"
stat_values = "mean"

# smoothing detrended data
win_len = 13
polyorder = 3

meta_columns = ["MoleculeID", "strain"]

schema = detrended_data.schema
detrended_data = (detrended_data
                .filter(pl.col("feature") == "CalcResultI")
                .select([well_columns, tpoint_col, "feature", stat_values])
                .group_by("Well")
                .map_groups(
                    lambda g: 
                        g.with_columns(
                            pl.Series(
                                stat_values,
                                savgol_filter(
                                    g[stat_values].to_numpy(),
                                    window_length=win_len,
                                    polyorder=polyorder
                                )
                            )
                        ),
                    schema=schema
                ).sort([well_columns, tpoint_col])
                .join(platemap.lazy(), on='Well', how='left')
                .sort(["strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, False, False])
            ).collect().to_pandas()

tpoints = [float(t) for t in sorted(detrended_data[tpoint_col].unique())]

cosinor_long = cosinor_fitted_long(cosinor_data, tpoints, tpoint_col).to_pandas()

cosinor_data = cosinor_data.collect().to_pandas()

detrended_data['group_label'] = detrended_data[meta_columns].apply(
    lambda row: ' | '.join([str(v) for v in row]), axis=1
)
cosinor_long['group_label'] = cosinor_long[meta_columns].apply(
    lambda row: ' | '.join([str(v) for v in row]), axis=1
)
cosinor_data['group_label'] = cosinor_data[meta_columns].apply(
    lambda row: ' | '.join([str(v) for v in row]), axis=1
)

# fitted_long.write_csv("/home/derfelt/PartchLabFiles/Circadian "
#                 "Rhythms/Live Study/enVision/cosinor_anal"
#                 "ysis/2026-05-11._.RESULTS-LONG_CSV.csv")

display(cosinor_data)

/tmp/ipykernel_2888/3498944735.py:23: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = detrended_data.schema


,Well,feature,stat,baseline,amplitude,phase,phase_hours,p_value,f_stat,ss_model,ss_null,damping_lambda,tau,fitted,MoleculeID,strain,sample_type,p_adjusted,group_label
0,A01,CalcResultI,mean,-0.019153,5.282105,0.268060,1.023915,0.105066,2.075936,20938.116496,21670.692799,0.019666,50.850109,"[5.074309100913683, 5.162145353769223, 5.15998...",NaN,NaN,NaN,0.156317,nan | nan
1,A02,CalcResultI,mean,-0.643287,20.917319,1.966489,7.511433,0.108355,2.051655,21756.483027,22508.788762,0.150776,6.632371,"[-8.705805907227898, -5.720019464973489, -3.04...",NaN,NaN,NaN,0.160235,nan | nan
2,A03,CalcResultI,mean,-0.384478,5.824582,0.876355,3.347428,0.003483,4.701853,25502.927701,27523.899848,0.005624,177.813550,"[3.3430007940361377, 3.883286820005832, 4.3476...",NaN,NaN,NaN,0.007423,nan | nan
3,A04,CalcResultI,mean,-0.364115,3.197838,5.955661,22.748948,0.040723,2.814159,18966.508231,19866.083079,0.000001,999999.998563,"[2.663731641131392, 2.503548361706514, 2.29429...",NaN,NaN,NaN,0.068058,nan | nan
4,A05,CalcResultI,mean,-0.099431,2.295610,5.742513,21.934786,0.426409,0.932116,23459.508141,23828.052762,0.002527,395.749168,"[1.8687398549089604, 1.6954059104426562, 1.491...",NaN,NaN,NaN,0.496630,nan | nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,P20,CalcResultI,mean,-0.205251,3.207369,1.258578,4.807416,0.294751,1.245498,32826.092744,33515.162984,0.003873,258.227360,"[0.7799560402177437, 1.1672733649134654, 1.529...",NaN,NaN,NaN,0.361403,nan | nan
373,P21,CalcResultI,mean,-0.490779,2.054113,1.158006,4.423256,0.484473,0.819858,28694.907092,29091.408568,0.000001,999999.985856,"[0.33326299450581026, 0.5718078032583662, 0.79...",NaN,NaN,NaN,0.546011,nan | nan
374,P22,CalcResultI,mean,-0.198548,5.308251,3.171227,12.113193,0.864448,0.245516,27340.508231,27453.640692,0.190846,5.239837,"[-5.5044679350434365, -4.998967602920768, -4.4...",NaN,NaN,NaN,0.876359,nan | nan
375,P23,CalcResultI,mean,0.304554,4.156442,6.102452,23.309650,0.201482,1.557160,32086.407155,32928.491582,0.007580,131.931630,"[4.393296384952476, 4.245832185328843, 4.03224...",NaN,NaN,NaN,0.268643,nan | nan


## Visualizing

In [4]:
def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def get_shades(base_color: str, n: int) -> list:
    """light → dark shades of base_color"""
    rgb = mcolors.to_rgb(base_color)
    return [tuple(1 - (1 - c) * (i + 1) / n for c in rgb) for i in range(n)]

def get_grays(n: int) -> list:
    return [str(round(0.3 + 0.6 * i / max(n - 1, 1), 2)) for i in range(n)]

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from tqdm import tqdm

# out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
#                 "Rhythms/Live Study/enVision/enVision_vis"
#                 "ual_results/cosinor_fitted_per_well.pdf")

out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/enVision_vi"
                "sual_results/2026-05-11")
out_file = out_file / "cosinor_fitted_per_well.pdf"

with PdfPages(out_file) as pdf:
    for well in tqdm(wells, desc="Well procesing"):
        
        well_detrended_df = (detrended_data[detrended_data[well_columns] == well]).sort_values(tpoint_col)
        well_cosinor = cosinor_data[cosinor_data[well_columns] == well]
        
        if well_cosinor.empty or well_detrended_df.empty:
            continue
        
        timepoints = sorted(well_detrended_df[tpoint_col].values)
        # print(timepoints)
        
        fitted = well_cosinor['fitted'].to_numpy()[0]
        # print(fitted)
        
        # display(well_detrended_df, well_cosinor)
        
        fig, ax = plt.subplots(figsize=(12,8))
        
        sns.lineplot(data=well_detrended_df, x=tpoint_col, y=stat_values, ax=ax, color='royalblue', linewidth=2, label='detrended data')
        ax.plot(timepoints, fitted, color='crimson', linewidth=2, linestyle='--', label='cosinor fit', zorder=10)
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        
        add_period_marks(ax, min(timepoints), max(timepoints))
        
        mol = str(well_cosinor['MoleculeID'].values[0])
        strain = str(well_cosinor['strain'].values[0])
        
        row = well_cosinor.iloc[0].to_dict()
        
        # ax.set_title(f"{'Well:':<15} {well} - {mol} | {strain}\n"
        #              f"{'Baseline:':<15} {row['baseline']:.4f}\n"
        #              f"{'Amplitude:':<15} {row['amplitude']:.4f}\n"
        #              f"{'Phase:':<15} {row['phase']:.4f}",
        #              loc='center',
        #              fontsize=14,
        #              fontfamily='monospace'
        #              )
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel('Expression')
        ax.legend(fontsize=8)
        
        fig.suptitle(
            f"Well: {well} - {mol} | {strain}",
            fontsize=12,
            fontfamily='monospace',
        )
        stats_text = (
            f"Baseline: {row['baseline']:.4f}    "
            f"Amplitude: {row['amplitude']:.4f}    "
            f"Phase: {row['phase_hours']:.4f} hours    "
            f"Damping factor: {row['damping_lambda']:.4f}    "
            f"p_adj: {row['p_adjusted']:.4e}    "
        )

        ax.set_title(stats_text, fontfamily='monospace', fontsize=10)

        # plt.subplots_adjust(top=0.88)  # make room for the two title lines
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well procesing: 100%|██████████| 377/377 [00:29<00:00, 12.90it/s]


### Replicate / group plots

In [22]:
out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/enVision_vi"
                "sual_results/2026-05-11")
out_file = out_file / "cosinor_replicate_results.pdf"

metric_cols = ['baseline', 'amplitude','phase_hours', 'p_adjusted']

with PdfPages(out_file) as pdf:
    groups = detrended_data['group_label'].unique()
    
    for group in tqdm(groups, 'plotting groups'):
        gdf_detrended = detrended_data[detrended_data['group_label'] == group]
        gdf_fitted = cosinor_long[cosinor_long['group_label'] == group]
        gdf_cosinor = cosinor_data[cosinor_data['group_label'] == group]
        
        fig, ax = plt.subplots(figsize=(14,8))
        
        # detrended signal
        sns.lineplot(gdf_detrended, x=tpoint_col, y=stat_values,
                     errorbar='sd', color='royalblue', linewidth=1.5,
                     label="detrended data", ax=ax, err_kws={
                         'alpha': 0.2, 'facecolor': 'royalblue'
                     })
        
        # fitted curve
        if not gdf_fitted.empty:
            sns.lineplot(gdf_fitted,
                         x=tpoint_col, y='fitted',
                         errorbar='sd', color='crimson', linewidth=2,
                         linestyle='--', label='cosinor fitted curve', ax=ax,
                         err_kws={'alpha':0.2, 'facecolor':'crimson'})
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_title(group)
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel('Expression')
        ax.legend(fontsize=8)
        
        add_period_marks(ax, gdf_detrended[tpoint_col].min(), gdf_detrended[tpoint_col].max())
        
        # metrics
        if not gdf_cosinor.empty:
            metric_lines = [f"n = {len(gdf_cosinor)} replicates"]
            for col in metric_cols:
                if col in gdf_cosinor.columns:
                    mean = gdf_cosinor[col].mean()
                    std  = gdf_cosinor[col].std()
                    metric_lines.append(f"{col:<25} {mean:.4f} ± {std:.4f}")
            
            # reporting time constant
            
            tau_mean = gdf_cosinor['tau'].mean()
            tau_std = gdf_cosinor['tau'].std()
            metric_lines.append(f"{'time constant (hrs)':<25} {tau_mean:.4f} ± {tau_std:.4f}")
            
            metrics_text = '\n'.join(metric_lines)
            ax.text(
                0.75, 0.13, metrics_text,
                transform=ax.transAxes,
                fontsize=8,
                fontfamily='monospace',
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
            )
        
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

plotting groups: 100%|██████████| 46/46 [00:09<00:00,  5.03it/s]


## We need to do one for certain group concentrations

In [33]:
PLOT_CONFIGS = [
    {
        'title': 'DMSO Controls',
        'drugs': [],
        'dmso' : ['0.25% DMSO', '0.125% DMSO', '0.0625% DMSO',
                  '0.03125% DMSO', '0.0156% DMSO', '0.0078% DMSO'],
    },
    {
        'title': 'MK-8692',
        'drugs': [('MK-8692', 'steelblue')],
        'dmso' : ['0.25% DMSO', '0.125% DMSO', '0.0625% DMSO',
                  '0.03125% DMSO', '0.0156% DMSO', '0.0078% DMSO'],
    },
    {
        'title': 'MK-8800',
        'drugs': [('MK-8800', 'darkorange')],
        'dmso': ['0.25% DMSO', '0.125% DMSO', '0.0625% DMSO',
                  '0.03125% DMSO', '0.0156% DMSO', '0.0078% DMSO'],
    }
    
]

well_columns = "Well"

tpoint_col = "Timepoint"
stat_values = "mean"

strains = cosinor_data['strain'].dropna().unique()

def plot_config_groups(ax, 
            detrended_df, 
            fitted_df, 
            config, 
            strain = None, 
            colored_dmso = True, 
            fitted = True, 
            plot_controls = True,
            distince_colors = False):
    d_df = detrended_df.copy()
    f_df = fitted_df.copy()
    
    if strain:
        d_df = d_df[d_df['strain'] == strain]
        f_df = f_df[f_df['strain'] == strain]
    
    
    def _plot_group(mol_mask_d, mol_mask_f, color, label, ax, linewidth=1.5, zorder=5):
        sub_d = d_df[mol_mask_d]
        sub_f = f_df[mol_mask_f]
        # display(sub_d, sub_f)
        
        if sub_d.empty:
            return
        
        # detrended data
        sns.lineplot(
            data=sub_d, x=tpoint_col, y=stat_values,
            errorbar='sd', color=color, linewidth=linewidth,
            linestyle='-', label=f'{label} (raw)', ax=ax,
            zorder=zorder, err_kws={'alpha': 0.1, 'facecolor': color}
        )
        
        # fitted curve
        if not sub_f.empty and fitted:
            sns.lineplot(
                data=sub_f, x=tpoint_col, y='fitted',
                errorbar='sd', color=color, linewidth=linewidth,
                linestyle='--', label=f"{label} (fit)", ax=ax,
                zorder=zorder+1, err_kws={'alpha':0.1, 'facecolor':color}
            )
    
    
    dmso_list = config['dmso']
    
    if not config['drugs'] and colored_dmso:
        palette = sns.color_palette("tab10", len(dmso_list))
        for dmso, color in zip(dmso_list, palette):
            _plot_group(
                d_df['MoleculeID'] == dmso,
                f_df['MoleculeID'] == dmso,
                color=color, label=dmso, ax=ax, linewidth=2, zorder=10
            )
    else:
        for drug, base_color in config['drugs']:
            drug_d = d_df[d_df['MoleculeID'].str.startswith(drug, na=False)]
            drug_f = f_df[f_df['MoleculeID'].str.startswith(drug, na=False)]
            concentrations = natsorted(drug_d['MoleculeID'].unique().tolist())
            
            if distince_colors:
                shades = sns.color_palette("tab10", len(concentrations))
            else:
                shades = get_shades(base_color, len(concentrations))
            
            for conc, shade in zip(concentrations, shades):
                _plot_group(
                    d_df['MoleculeID'] == conc,
                    f_df['MoleculeID'] == conc,
                    color=shade, label=conc, ax=ax
                )
        
        if plot_controls:
        # DMSOs    
            grays = get_grays(len(dmso_list))
            for dmso, gray in zip(dmso_list, grays):
                _plot_group(
                    d_df['MoleculeID'] == dmso,
                    f_df['MoleculeID'] == conc,
                    color=gray, label=dmso, ax=ax, linewidth=2, zorder=10
                )
        
    
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    title = config['title']
    if strain:
        title += f' - {strain}'
        
    
    ax.set_title(title)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Expression')
    ax.legend(fontsize=6, loc='upper right', ncol=2)
    add_period_marks(ax, d_df[tpoint_col].min(), d_df[tpoint_col].max())


In [34]:
out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/enVision_vi"
                "sual_results/2026-05-11")
out_file = out_file / "cosinor_grouped_analysis_no_controls.pdf"

with PdfPages(out_file) as pdf:
    for config in tqdm(PLOT_CONFIGS, desc='plot configs'):
        fig, axs = plt.subplots(2, 1, figsize=(24,10), sharey=False)
        for ax, strain in zip(axs, strains):
            plot_config_groups(ax=ax, 
                               detrended_df=detrended_data,
                               fitted_df=cosinor_long,
                               config=config,
                               strain=strain,
                               colored_dmso=True,
                               fitted=False,
                               plot_controls=False,
                               distince_colors=True)
            
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

plot configs:   0%|          | 0/3 [00:00<?, ?it/s]

plot configs: 100%|██████████| 3/3 [00:02<00:00,  1.02it/s]
